# Fetch ACM conference papers by topic

Finds papers from ACM networking venues (default: SIGCOMM, CoNEXT, IMC, SIGMETRICS) over the last K years (default 10) that match the topics in `to_search.md`, and writes them to a CSV.

### Topics live in `to_search.md`
One topic per `- bullet`. A paper is kept if any topic is found.

### Three-stage match (in precedence order)
For each paper we look for each topic in, in order:
1. **CCS** — the paper's **OpenAlex concepts** (closest available proxy for ACM CCS concepts)
2. **Keywords** — the paper's **OpenAlex keywords**
3. **Additional** — the paper's **title + abstract** text

The CSV records *where* each topic matched (`matched_ccs`, `matched_keywords`, `matched_additional`) and is sorted with CCS-concept matches first.

### Data sources (two free APIs, no keys)
- **[Crossref](https://api.crossref.org/)** — complete paper list per conference/year (title, **DOI**, **URL**, year, authors).
- **[OpenAlex](https://docs.openalex.org/)** — queried by DOI for abstract + concepts + keywords.

> ⚠️ The official ACM **CCS Concepts** and **Author Keywords** fields are only on ACM DL paper pages, which block automated access (HTTP 403). There is no public API for them. So "CCS" here means OpenAlex concepts, "Keywords" means OpenAlex keywords — a practical proxy, not the literal ACM CCS taxonomy.

Only dependency is `requests` (`pandas` optional, for the preview).

In [ ]:
# If needed:  pip install requests pandas
import requests, time, csv, json, re
from datetime import datetime

# ====================================================================
# CONFIG  -- edit this cell (and edit to_search.md for the topics)
# ====================================================================

EXTRACTION_FILE = "to_search.md"   # topics/keywords to search for (one per "- bullet")

# Conferences to search. Each label maps to:
#   query : a Crossref container-title relevance query
#   match : a regex the actual container-title must contain (filters noise)
CONFERENCES = {
    "SIGCOMM":    {"query": "ACM SIGCOMM Conference",
                    "match": r"SIGCOMM"},
    "CoNEXT":     {"query": "emerging Networking EXperiments and Technologies",
                    "match": r"CoNEXT|emerging Networking EXperiments"},
    "IMC":        {"query": "ACM Internet Measurement Conference",
                    "match": r"Internet Measurement Conference|Internet Measurement Workshop"},
    "SIGMETRICS": {"query": "ACM SIGMETRICS Measurement Modeling Computer Systems",
                    "match": r"SIGMETRICS|Measurement and Modeling of Computer Systems"},
}

K_YEARS = 10                 # how many years back from today
REQUIRE_TAG_MATCH = True     # False -> keep all papers, ignore the term list
INCLUDE_WORKSHOPS = True     # workshops/companion/posters get flagged, not dropped
ENRICH_WITH_OPENALEX = True  # must stay True: concepts/keywords/abstract come from here

# APIs ask for an email to join their fast "polite pool". Use your own.
MAILTO = "you@example.org"

OUTPUT_CSV = "acm_papers.csv"

THIS_YEAR = datetime.now().year
FROM_YEAR = THIS_YEAR - K_YEARS + 1
print(f"Conferences: {list(CONFERENCES)}")
print(f"Years      : {FROM_YEAR}..{THIS_YEAR}  (K={K_YEARS})")

In [ ]:
# ---- Load search terms from extraction_words.md --------------------
def load_terms(path):
    """Return (include_terms, exclude_terms). Bullets under a header that
    contains 'exclusion' are exclude terms; all other bullets are includes."""
    include, exclude = [], []
    in_exclusion = False
    with open(path, encoding="utf-8") as f:
        for raw in f:
            s = raw.strip()
            if s.startswith("#"):                      # markdown header
                in_exclusion = "exclusion" in s.lower()
                continue
            if s.startswith("- "):
                term = s[2:].split("  #")[0].strip()   # drop inline comments
                if term:
                    (exclude if in_exclusion else include).append(term)
    def dedupe(seq):
        seen, out = set(), []
        for t in seq:
            if t.lower() not in seen:
                seen.add(t.lower()); out.append(t)
        return out
    return dedupe(include), dedupe(exclude)


def compile_terms(terms):
    """Compile each term to a word-boundary regex. ALL-CAPS acronyms are
    matched case-sensitively so they don't hit common lowercase words."""
    pats = []
    for t in terms:
        if re.fullmatch(r"[A-Z0-9/+\-]{2,6}", t):      # acronym, e.g. TCP, HTTP/3
            pats.append((t, re.compile(r"(?<![A-Za-z0-9])" + re.escape(t)
                                       + r"(?![A-Za-z0-9])")))
        else:
            pats.append((t, re.compile(r"\b" + re.escape(t) + r"\b", re.I)))
    return pats


def find_terms(text, pats):
    return [t for t, p in pats if p.search(text)]


INCLUDE_TERMS, EXCLUDE_TERMS = load_terms(EXTRACTION_FILE)
INCLUDE_PATS = compile_terms(INCLUDE_TERMS)
EXCLUDE_PATS = compile_terms(EXCLUDE_TERMS)
print(f"Loaded {len(INCLUDE_TERMS)} include terms, {len(EXCLUDE_TERMS)} exclusion terms")
print("First includes:", INCLUDE_TERMS[:8])
print("Exclusions    :", EXCLUDE_TERMS)

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": f"acm-paper-fetch/1.0 (mailto:{MAILTO})"})


def _get(url, params=None, retries=4):
    """GET with polite-pool mailto + simple retry/backoff."""
    params = dict(params or {})
    params["mailto"] = MAILTO
    for attempt in range(retries):
        try:
            r = SESSION.get(url, params=params, timeout=60)
            if r.status_code == 200:
                return r.json()
            if r.status_code in (429, 500, 502, 503):
                time.sleep(2 * (attempt + 1)); continue
            r.raise_for_status()
        except requests.RequestException:
            if attempt == retries - 1:
                raise
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"GET failed after {retries} tries: {url}")


def strip_tags(x):
    return re.sub(r"<[^>]+>", " ", x or "").strip()

## Step 1 — enumerate papers from Crossref
Crossref's `query.container-title` is a relevance search, so the conference's own papers cluster at the top. We cursor-page (200/request) and **stop after a few consecutive pages with no container-title match** — that captures every real paper without paging through all of Crossref. (The previous single 1000-row request silently truncated busy years.)

In [ ]:
CROSSREF = "https://api.crossref.org/works"
WORKSHOP_RE = re.compile(r"workshop|companion|student|poster|tutorial|demo|doctoral", re.I)
MAX_MISS_PAGES = 3      # stop after this many consecutive all-miss pages
MAX_SCAN = 6000         # hard safety cap on items scanned per conf/year


def crossref_authors(item):
    out = []
    for a in item.get("author", []) or []:
        name = " ".join(p for p in [a.get("given"), a.get("family")] if p)
        out.append(name or a.get("name", ""))
    return [n for n in out if n]


def fetch_conf_year(conf, cfg, year):
    """Return base records for one conference and year from Crossref."""
    match_re = re.compile(cfg["match"], re.I)
    rows, cursor, miss_pages, scanned = [], "*", 0, 0
    while cursor and miss_pages < MAX_MISS_PAGES and scanned < MAX_SCAN:
        data = _get(CROSSREF, {
            "query.container-title": cfg["query"],
            "filter": (f"type:proceedings-article,"
                        f"from-pub-date:{year}-01-01,until-pub-date:{year}-12-31"),
            "rows": 200,
            "cursor": cursor,
            "select": "DOI,title,container-title,published,URL,author",
        })
        items = data["message"]["items"]
        if not items:
            break
        page_hits = 0
        for it in items:
            scanned += 1
            ct = (it.get("container-title") or [""])[0]
            if not match_re.search(ct):
                continue
            page_hits += 1
            is_ws = bool(WORKSHOP_RE.search(ct))
            if is_ws and not INCLUDE_WORKSHOPS:
                continue
            title = strip_tags((it.get("title") or [""])[0])
            if not title:
                continue
            doi = (it.get("DOI") or "").lower()
            rows.append({
                "title": title, "year": year, "conference": conf, "venue": ct,
                "doi": doi,
                "url": it.get("URL") or (f"https://doi.org/{doi}" if doi else ""),
                "authors": "; ".join(crossref_authors(it)),
                "is_workshop": is_ws,
            })
        miss_pages = miss_pages + 1 if page_hits == 0 else 0
        cursor = data["message"].get("next-cursor")
        time.sleep(0.15)
    if scanned >= MAX_SCAN:
        print(f"    [warn] {conf} {year}: hit MAX_SCAN={MAX_SCAN}; raise it if needed.")
    return rows

In [ ]:
base = {}  # doi -> record  (dedupe across years/confs)
for conf, cfg in CONFERENCES.items():
    n_conf = 0
    for year in range(FROM_YEAR, THIS_YEAR + 1):
        for r in fetch_conf_year(conf, cfg, year):
            if r["doi"] and r["doi"] not in base:
                base[r["doi"]] = r
                n_conf += 1
    print(f"[ok] {conf}: {n_conf} papers {FROM_YEAR}-{THIS_YEAR}")

records = list(base.values())
print(f"\nTotal unique papers from Crossref: {len(records)}")

## Step 2 — enrich with OpenAlex (abstracts + concept tags)
Batched 50 DOIs per request. Papers not in OpenAlex keep an empty abstract/concepts and are matched on title only.

In [ ]:
OPENALEX = "https://api.openalex.org/works"


def reconstruct_abstract(inv):
    if not inv:
        return ""
    pos = [(i, w) for w, idxs in inv.items() for i in idxs]
    pos.sort()
    return " ".join(w for _, w in pos)


def chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


enrich = {}  # doi -> {abstract, concepts, keywords}
if ENRICH_WITH_OPENALEX:
    all_dois = [r["doi"] for r in records if r["doi"]]
    for batch in chunks(all_dois, 50):
        data = _get(OPENALEX, {
            "filter": "doi:" + "|".join(batch),
            "per_page": 50,
            "select": "doi,concepts,keywords,abstract_inverted_index",
        })
        for w in data.get("results", []):
            doi = (w.get("doi") or "").replace("https://doi.org/", "").lower()
            enrich[doi] = {
                "abstract": reconstruct_abstract(w.get("abstract_inverted_index")),
                "concepts": [c["display_name"] for c in (w.get("concepts") or [])],
                "keywords": [k.get("display_name", "") for k in (w.get("keywords") or [])],
            }
        time.sleep(0.2)
    print(f"Enriched {len(enrich)}/{len(all_dois)} papers via OpenAlex")
else:
    print("OpenAlex enrichment disabled; matching on title only.")

## Step 3 — match terms, filter, and write the CSV
Sorted by number of matched terms (strongest candidates first).

In [ ]:
# Three-stage match per the requested precedence:
#   1) CCS        -> OpenAlex concepts   (closest available proxy for ACM CCS concepts)
#   2) keywords   -> OpenAlex keywords
#   3) additional -> title + abstract text
# A paper is kept if a topic from to_search.md is found in ANY of the three.

def field_matches(text, pats):
    return find_terms(text, pats)


final = []
for r in records:
    e = enrich.get(r["doi"], {})
    abstract = e.get("abstract", "")
    concepts = e.get("concepts", [])
    keywords = e.get("keywords", [])

    m_ccs = field_matches(" ; ".join(concepts), INCLUDE_PATS)
    m_kw = field_matches(" ; ".join(keywords), INCLUDE_PATS)
    m_add = field_matches(r["title"] + " . " + abstract, INCLUDE_PATS)

    all_matched = list(dict.fromkeys(m_ccs + m_kw + m_add))  # distinct, order-preserving
    if REQUIRE_TAG_MATCH and INCLUDE_TERMS and not all_matched:
        continue

    final.append({
        **r,
        "n_matched": len(all_matched),
        "matched_ccs": "; ".join(m_ccs),
        "matched_keywords": "; ".join(m_kw),
        "matched_additional": "; ".join(m_add),
        "matched_terms": "; ".join(all_matched),
        "openalex_concepts": "; ".join(concepts),
        "openalex_keywords": "; ".join(keywords),
        "abstract": abstract,
    })

# Strongest first: CCS hits, then keyword hits, then additional hits, then recency.
final.sort(key=lambda r: (-len(r["matched_ccs"].split("; ")) if r["matched_ccs"] else 0,
                          -len(r["matched_keywords"].split("; ")) if r["matched_keywords"] else 0,
                          -r["n_matched"], -(r["year"] or 0)))

FIELDS = ["title", "year", "conference", "venue", "doi", "url",
          "n_matched", "matched_ccs", "matched_keywords", "matched_additional",
          "matched_terms", "authors", "is_workshop",
          "openalex_concepts", "openalex_keywords", "abstract"]

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=FIELDS)
    w.writeheader()
    w.writerows(final)

print(f"Kept {len(final)} of {len(records)} papers -> {OUTPUT_CSV}")
print(f"  with a CCS-concept (OpenAlex) match : {sum(1 for r in final if r['matched_ccs'])}")
print(f"  with a keyword match               : {sum(1 for r in final if r['matched_keywords'])}")
print(f"  with a title/abstract match        : {sum(1 for r in final if r['matched_additional'])}")

In [ ]:
# Optional preview (needs pandas: pip install pandas)
try:
    import pandas as pd
    df = pd.read_csv(OUTPUT_CSV)
    print("By conference:\n", df["conference"].value_counts(), sep="")
    print("\nBy year:\n", df["year"].value_counts().sort_index(), sep="")
    display(df[["n_matched", "year", "conference", "title",
                "matched_ccs", "matched_keywords"]].head(25))
except ImportError:
    for r in final[:25]:
        print(f"[{r['n_matched']}] {r['year']} {r['conference']:<10} {r['title'][:60]}")

## Customizing
* **Search terms** — edit `extraction_words.md` (one term per `- bullet`). Re-run from the term-loading cell.
* **Keep everything (no filter)** — set `REQUIRE_TAG_MATCH = False`.
* **Main conference only** — set `INCLUDE_WORKSHOPS = False` (otherwise workshop/poster papers are flagged in `is_workshop`).
* **Add a conference** — add an entry to `CONFERENCES`: a `query` matching its proceedings container-title and a specific `match` regex. Verify with the `venue` column.
* **Triage output** — the CSV is sorted by `n_matched` (strongest candidates first). `matched_terms` shows why each was kept; `possible_out_of_scope` flags likely-unsupported topics from `capabilities.md` §4.
* **Real ACM CCS concepts** — only on ACM DL pages (no public API); this uses OpenAlex concepts + keyword text as the practical free proxy.